# MVP training run

CIFAR-100 + pretrained ResNet-18, logged to Aim. Single-model placeholder
where the multi-agent system will go later.

To view runs: `aim up` in a terminal from this directory, then open localhost:43800.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from aim import Run

from train import train
from eval import evaluate, plot_loss, plot_metrics

## config

In [ ]:
cfg = {
  "batch_size": 128,
  "epochs": 2,
  "lr": 1e-3,
  "weight_decay": 1e-4,
  "image_size": 224,
  "num_classes": 100,
  "seed": 0,
}

In [ ]:
torch.manual_seed(cfg["seed"])
torch.cuda.manual_seed_all(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

## data

ImageNet normalization since we use an ImageNet-pretrained backbone.

In [ ]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
  transforms.Resize(cfg["image_size"]),
  transforms.RandomHorizontalFlip(),
  transforms.ToTensor(),
  transforms.Normalize(mean, std),
])
test_tf = transforms.Compose([
  transforms.Resize(cfg["image_size"]),
  transforms.ToTensor(),
  transforms.Normalize(mean, std),
])

train_set = datasets.CIFAR100("./data", train=True, download=True, transform=train_tf)
test_set = datasets.CIFAR100("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=cfg["batch_size"], shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=cfg["batch_size"], shuffle=False, num_workers=2)

print(f"train: {len(train_set)}, test: {len(test_set)}")

## model

Swap this cell for the multi-agent system when ready.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, cfg["num_classes"])
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_params:,}")

## optimizer + loss

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
criterion = nn.CrossEntropyLoss()

## aim run

Hparams logged for filtering/comparing runs later.

In [ ]:
run = Run(experiment="mvp")
run["hparams"] = cfg
run["hparams", "model"] = "resnet18-pretrained"
print(f"aim run hash: {run.hash}")

## train

In [ ]:
losses = train(
  model=model,
  loader=train_loader,
  optimizer=optimizer,
  criterion=criterion,
  epochs=cfg["epochs"],
  device=device,
  run=run,
  log_every=50,
)

## loss curve

In [ ]:
plot_loss(losses, smooth=50)

## evaluate on test

In [ ]:
results = evaluate(model, test_loader, device=device, criterion=criterion)
print(results)
for k, v in results.items():
  run.track(v, name=f"test_{k}")

## metrics

In [ ]:
plot_metrics(results)

## done

In [ ]:
run.close()